In [ ]:
# !pip install -q gradio pandas
# !mkdir -p ratings_interface/test_images

In [ ]:
# import os
# import json
# import hashlib
# import itertools
# from pathlib import Path
# from typing import List
# from PIL import Image, ImageDraw, ImageFont
# from tqdm.auto import tqdm
# import random

# # --- Configuration ---
# # CRITICAL CHANGE: Use the specific directory for GUI testing
# BASE_OUTPUT_DIR = Path("ratings_interface/test_images") 
# NUM_SAMPLES_PER_PROMPT = 2 # Always generate A and B for pairwise comparison
# NUM_PROMPTS_TO_GENERATE = 10 # Limit for a quick test run of the GUI

# # New list of distinct color PAIRS to cycle through for each new prompt ID
# COLOR_PAIRS = [
#     ("#3399FF", "#FF9933"), # Blue / Orange
#     ("#57C78D", "#D9534F"), # Green / Red
#     ("#B366FF", "#FFCC33"), # Purple / Yellow
#     ("#33CCCC", "#FF6666"), # Cyan / Coral
#     ("#FF80B3", "#80B3FF"), # Pink / Light Blue
# ]

# FONT_SIZE = 20

# # ======================================================================================
# #  This section is adapted from your '2_generate_prompts.ipynb' notebook
# # ======================================================================================
# def generate_prompts(age_increment: int = 5):
#     """Generates a list of prompts based on specified attributes."""
#     # Using your full prompt matrix combinations
#     ages = list(range(0, 90, age_increment))
#     genders = ["male", "female"]
#     ethnicities = ["caucasian", "afroamerican", "asian"] 
#     emotions = {
#         'neutral': "with a neutral expression",
#         'happy': "smiling happily",
#         'slightlyhappy': "smiling slightly",
#         'other': "showing a subtle emotion"
#     }

#     all_prompts = []
#     all_combinations = itertools.product(ages, genders, ethnicities, list(emotions.keys()))

#     for age, gender, ethnicity, emotion_key in all_combinations:
#         prompt_parts = [f"A {age} years old", ethnicity, gender, emotions[emotion_key]]
#         prompt = " ".join(prompt_parts)
#         all_prompts.append(prompt)
    
#     return all_prompts

# # ======================================================================================
# #  MOCK Sample Generator
# # ======================================================================================
# def generate_mock_samples(
#     prompt: str,
#     prompt_hash: str,
#     output_dir: Path,
#     num_samples: int,
#     color_pair: tuple # New argument: (COLOR_A, COLOR_B)
# ) -> List[dict]:
#     """
#     Generates MOCK samples for a single prompt and saves placeholder artifacts.
#     """
#     metadata_list = []
    
#     # Map the sample index to the specific color and letter for the current pair
#     COLOR_A, COLOR_B = color_pair
#     color_map = {0: (COLOR_A, 'A'), 1: (COLOR_B, 'B')}

#     for i in range(num_samples):
        
#         if i not in color_map:
#             continue 

#         background_color, sample_letter = color_map[i]
        
#         # Use a simple counter prefix for the GUI test phase
#         sample_id_prefix = prompt_hash[:5] 
#         sample_base_name = f"{sample_id_prefix}_{sample_letter}"

#         # 1. Create and save a placeholder image with color and text
#         img = Image.new('RGB', (512, 512), color=background_color)
#         draw = ImageDraw.Draw(img)
        
#         # Try to load a default font if available
#         try:
#             font = ImageFont.truetype("arial.ttf", FONT_SIZE) 
#         except IOError:
#             font = ImageFont.load_default()

#         # Text to display on the image
#         text_lines = [
#             f"ID: {sample_id_prefix}",
#             f"Color Pair: {COLOR_A} / {COLOR_B}", # Added color info to the image
#             f"Sample: {sample_letter} (Preferred/Rejected)",
#             f"Prompt: {prompt[:80]}..."
#         ]
        
#         y_text = 10
#         for line in text_lines:
#             draw.text((10, y_text), line, fill='white', font=font)
#             y_text += FONT_SIZE + 5

#         image_path = output_dir / f"{sample_base_name}.jpg"
#         img.save(image_path)

#         # 2. Create an empty placeholder latent file
#         latent_path = output_dir / f"{sample_base_name}_latent.pt"
#         latent_path.touch() 

#         # 3. Create metadata dictionary for this sample
#         seed = random.randint(0, 2**32 - 1)
#         metadata = {
#             "prompt_hash": prompt_hash,
#             "prompt": prompt,
#             "seed": seed,
#             "sample_id": sample_base_name,
#             "image_path": str(image_path),
#             "latent_path": str(latent_path),
#         }
#         metadata_list.append(metadata)
        
#     return metadata_list

# def generate_mock_test_dataset(
#     prompts: List[str],
#     base_output_dir: Path,
#     num_samples_per_prompt: int = 2,
# ):
#     """Orchestrates the generation of the MOCK test dataset, cycling through color pairs."""
#     print(f"✨ Starting MOCK GUI data generation for {len(prompts)} prompts...")
#     # Ensure the full path exists
#     base_output_dir.mkdir(parents=True, exist_ok=True)
    
#     all_metadata = []
    
#     # Iterate through all selected prompts
#     for i, prompt in enumerate(tqdm(prompts, desc="Creating Mock Images")):
        
#         # Get the color pair based on the current prompt index
#         color_pair = COLOR_PAIRS[i % len(COLOR_PAIRS)]
        
#         prompt_hash = hashlib.sha256(prompt.encode()).hexdigest()[:10]
        
#         generated_metadata = generate_mock_samples(
#             prompt=prompt,
#             prompt_hash=prompt_hash,
#             output_dir=base_output_dir,
#             num_samples=num_samples_per_prompt,
#             color_pair=color_pair # Pass the cycled color pair
#         )
#         all_metadata.extend(generated_metadata)
    
#     # Save the master manifest file in the parent 'ratings_interface' directory
#     manifest_path = base_output_dir.parent / "generation_manifest_test.jsonl"
#     with open(manifest_path, 'w') as f:
#         for meta_item in all_metadata:
#             f.write(json.dumps(meta_item) + '\n')
            
#     print("\n" + "="*50)
#     print(f"✅ MOCK GUI Setup Complete with Cycled Color Pairs!")
#     print(f"Total prompt pairs created: {len(all_metadata) / 2}")
#     print(f"Data saved in: {base_output_dir}")
#     print("="*50)

# if __name__ == "__main__":
#     # 1. Generate the full list of prompts
#     print("Step 1: Generating all possible prompts from matrix...")
#     prompts = generate_prompts(age_increment=5)
    
#     if NUM_PROMPTS_TO_GENERATE is not None:
#         print(f"\nLimiting generation to {NUM_PROMPTS_TO_GENERATE} prompts for this test run.")
#         prompts_to_run = prompts[:NUM_PROMPTS_TO_GENERATE]
#     else:
#         prompts_to_run = prompts
    
#     # 2. Generate the mock dataset from these prompts
#     print("\nStep 2: Creating mock colored images, latents, and metadata manifest...")
#     generate_mock_test_dataset(
#         prompts=prompts_to_run,
#         base_output_dir=BASE_OUTPUT_DIR,
#         num_samples_per_prompt=NUM_SAMPLES_PER_PROMPT
#     )

In [ ]:
import gradio as gr
import os
import json
import time
from datetime import datetime

IMAGE_DIR = "ratings_interface/test_images"
RATINGS_FILE = "ratings_interface/ratings.jsonl"

# --- 1. Helper Functions ---

def find_image_pairs(directory):
    """Scans a directory and groups images by prefix (e.g., '00001_A.jpg', '00001_B.jpg')."""
    pairs = {}
    for filename in os.listdir(directory):
        if filename.endswith((".jpg", ".png")):
            prefix = filename.split('_')[0]
            if prefix not in pairs:
                pairs[prefix] = {}
            # Use 'A' and 'B' keys to store paths
            key = "A" if "_A" in filename else "B"
            pairs[prefix][key] = os.path.join(directory, filename)
    
    # Filter for valid pairs and ensure both A and B are present
    pair_list = [{"id": prefix, **paths} for prefix, paths in pairs.items() if "A" in paths and "B" in paths]
    return sorted(pair_list, key=lambda x: x['id'])

def load_rated_ids(ratings_file):
    """Loads all prompt_ids from the ratings file to avoid re-rating."""
    rated_ids = set()
    if os.path.exists(ratings_file):
        with open(ratings_file, 'r') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    if 'prompt_id' in data:
                        rated_ids.add(data['prompt_id'])
                except (json.JSONDecodeError, KeyError):
                    print(f"Warning: Skipping corrupted line in ratings file: {line}")
    return rated_ids

# --- 2. Data Loading and Filtering ---

all_pairs = find_image_pairs(IMAGE_DIR)
rated_ids = load_rated_ids(RATINGS_FILE)

# Filter out the pairs that have already been rated
unrated_pairs = [p for p in all_pairs if p['id'] not in rated_ids]

print(f"Found {len(all_pairs)} total pairs in the directory.")
print(f"Found {len(rated_ids)} already rated pairs in '{RATINGS_FILE}'.")
print(f"🚀 Starting rating session with {len(unrated_pairs)} pairs remaining.")


# --- 3. Gradio App Logic (Updated) ---

def save_preference(pair_id, preferred_path, rejected_path):
    """Appends a PREFERRED rating to the JSON Lines file."""
    rating = {
        "prompt_id": pair_id,
        "preferred_img": os.path.basename(preferred_path),
        "rejected_img": os.path.basename(rejected_path),
        "rating_status": "PREFERRED",  # Flag for successful preference
        "timestamp": datetime.utcnow().isoformat()
    }
    with open(RATINGS_FILE, 'a') as f:
        f.write(json.dumps(rating) + '\n')
    print(f"Saved rating for ID {pair_id}: '{os.path.basename(preferred_path)}' was preferred.")

def save_skip(pair_id, path_a, path_b):
    """Appends a SKIPPED rating to the JSON Lines file."""
    rating = {
        "prompt_id": pair_id,
        # Store both paths as skipped for later analysis
        "rejected_img_A": os.path.basename(path_a),
        "rejected_img_B": os.path.basename(path_b),
        "rating_status": "SKIPPED_BAD_FIT", # Flag for skipped pair
        "timestamp": datetime.utcnow().isoformat()
    }
    with open(RATINGS_FILE, 'a') as f:
        f.write(json.dumps(rating) + '\n')
    print(f"Skipped pair ID {pair_id}. Both images flagged as bad fit.")

def get_next_pair(current_index):
    """Loads the next pair of images or ends the session if complete."""
    current_index += 1
    if current_index >= len(unrated_pairs):
        # All images have been rated
        completion_message = "✅ All pairs rated! Session complete. You can close this tab."
        print("\n" + completion_message)
        time.sleep(2)
        
        # We need to return gr.update(value=...) for the image components 
        # to clear them, or None to tell Gradio to use the default empty value.
        return None, None, completion_message, gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False), current_index
    
    pair = unrated_pairs[current_index]
    progress = f"Rating pair: {pair['id']} ({current_index + 1} of {len(unrated_pairs)})"
    return pair["A"], pair["B"], progress, gr.update(interactive=True), gr.update(interactive=True), gr.update(interactive=True), current_index

def choose_preference(current_index, choice):
    """Handler for choosing left or right (A or B)."""
    if 0 <= current_index < len(unrated_pairs):
        pair = unrated_pairs[current_index]
        if choice == "A":
            save_preference(pair['id'], pair['A'], pair['B'])
        else: # Choice is "B"
            save_preference(pair['id'], pair['B'], pair['A'])
    # Load the next pair regardless of choice
    return get_next_pair(current_index)

def choose_skip(current_index):
    """Handler for choosing to skip the pair."""
    if 0 <= current_index < len(unrated_pairs):
        pair = unrated_pairs[current_index]
        save_skip(pair['id'], pair['A'], pair['B'])
    # Load the next pair
    return get_next_pair(current_index)

# --- 4. UI Definition ---
with gr.Blocks() as demo:
    current_index = gr.State(value=-1)

    gr.Markdown("# Image Comparison for RLHF Rating")
    gr.Markdown("Click which image is better, or **Skip / Bad Pair** if neither is a good match for the prompt. The next pair will load automatically.")
    
    info_box = gr.Markdown("Loading first pair...")
        
    with gr.Row(equal_height=True):
        img_a = gr.Image(label="Image A", type="filepath")
        img_b = gr.Image(label="Image B", type="filepath")

    with gr.Row():
        left_button = gr.Button("⬅️ Left is Better", variant="primary", scale=1)
        skip_button = gr.Button("Skip / Bad Pair", scale=1)
        right_button = gr.Button("Right is Better ➡️", variant="primary", scale=1)

    # --- 5. Event Listeners (Updated) ---
    left_button.click(
        fn=lambda idx: choose_preference(idx, "A"),
        inputs=[current_index],
        outputs=[img_a, img_b, info_box, left_button, right_button, skip_button, current_index]
    )
    right_button.click(
        fn=lambda idx: choose_preference(idx, "B"),
        inputs=[current_index],
        outputs=[img_a, img_b, info_box, left_button, right_button, skip_button, current_index]
    )
    # The skip button now calls the new choose_skip function
    skip_button.click(
        fn=choose_skip,
        inputs=[current_index],
        outputs=[img_a, img_b, info_box, left_button, right_button, skip_button, current_index]
    )
    
    demo.load(
        fn=get_next_pair,
        inputs=[current_index],
        outputs=[img_a, img_b, info_box, left_button, right_button, skip_button, current_index]
    )

# --- 6. Launch the App ---
if len(unrated_pairs) == 0:
    print("✅ No unrated images found. Nothing to do.")
else:
    demo.launch(share=False, debug=True)